In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from pathlib import Path

import pandas as pd
from PIL import Image
from fiftyone.core.session.notebooks import display
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
class BeerYoloDataset(Dataset):
    """
    Очікує CSV з колонками:
        path | klass | x_min | y_min | x_max | y_max
    де координати — в пікселях.
    """

    PATH_PRIORITY = ("path", "aug_path", "cropped_path", "orig_path")

    def __init__(self, meta_csv: str | Path,
                 root: str | Path | None = None,
                 img_size: int = 448,
                 S: int = 7,
                 transform=None):

        self.meta_csv = Path(meta_csv)
        if not self.meta_csv.exists():
            raise FileNotFoundError(f"Metadata CSV not found: {self.meta_csv}")

        self.root = Path(root) if root is not None else Path(".")
        self.S = S
        self.img_size = img_size
        self.transform = transform

        df = pd.read_csv(self.meta_csv)

        required_cols = {"klass", "x_min", "y_min", "x_max", "y_max"}
        if not required_cols.issubset(df.columns):
            raise ValueError(f"CSV must contain columns: {required_cols}")

        candidates = self.PATH_PRIORITY
        available = tuple(c for c in candidates if c in df.columns)
        resolved_path = df[available[0]].copy()
        for col in available[1:]:
            resolved_path = resolved_path.fillna(df[col])
        df = df.assign(resolved_path=resolved_path)
        df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)
        df["resolved_path"] = df["resolved_path"].astype(str)

        self._records = df
        classes = sorted(df["klass"].unique())
        self.class_to_idx = {k: i for i, k in enumerate(classes)}
        self.idx_to_class = {i: k for k, i in self.class_to_idx.items()}

    def __len__(self):
        return len(self._records)

    def __getitem__(self, idx):
        row = self._records.iloc[idx]
        img_path = Path(self.root) / row["resolved_path"]
        image = Image.open(img_path).convert("RGB")

        W, H = image.size
        # перетворюємо bbox у нормалізовані координати YOLO
        x1, y1, x2, y2 = row[["x_min", "y_min", "x_max", "y_max"]]
        xc = ((x1 + x2) / 2) / W
        yc = ((y1 + y2) / 2) / H
        bw = (x2 - x1) / W
        bh = (y2 - y1) / H
        label = self.class_to_idx[row["klass"]]

        # будуємо мапу [S,S,5+C]
        target = torch.zeros(self.S, self.S, 5 + len(self.class_to_idx))
        cell_x = int(xc * self.S)
        cell_y = int(yc * self.S)
        target[cell_y, cell_x, 0:4] = torch.tensor([xc * self.S - cell_x,
                                                    yc * self.S - cell_y,
                                                    bw, bh])
        target[cell_y, cell_x, 4] = 1.0
        target[cell_y, cell_x, 5 + label] = 1.0

        if self.transform:
            image = self.transform(image)
        return image, target

In [ ]:
S = 7          # розмір сітки
B = 2          # кількість боксів на клітинку
C = 20         # кількість класів (PASCAL VOC)
IMG_SIZE = 448 # роздільна здатність під час тренування

λ_coord = 5.0
λ_noobj = 0.5

In [ ]:
def conv_bn_lrelu(in_ch, out_ch, k, s=1, p=None):
    if p is None:
        p = (k - 1) // 2
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.LeakyReLU(0.1, inplace=True),
    )

In [ ]:
class YOLOv1(nn.Module):
    def __init__(self, S=7, B=2, C=20):
        super().__init__()
        self.S, self.B, self.C = S, B, C

        self.features = nn.Sequential(
            conv_bn_lrelu(3,   64, 7, s=2, p=3),    
            nn.MaxPool2d(2,2),

            conv_bn_lrelu(64,  192, 3),
            nn.MaxPool2d(2,2),

            conv_bn_lrelu(192, 128, 1),
            conv_bn_lrelu(128, 256, 3),
            conv_bn_lrelu(256, 256, 1),
            conv_bn_lrelu(256, 512, 3),
            nn.MaxPool2d(2,2),

            conv_bn_lrelu(512, 256, 1),
            conv_bn_lrelu(256, 512, 3),
            conv_bn_lrelu(512, 256, 1),
            conv_bn_lrelu(256, 512, 3),
            conv_bn_lrelu(512, 256, 1),
            conv_bn_lrelu(256, 512, 3),
            conv_bn_lrelu(512, 256, 1),
            conv_bn_lrelu(256, 512, 3),

            conv_bn_lrelu(512, 512, 1),
            conv_bn_lrelu(512, 1024, 3),
            nn.MaxPool2d(2,2),

            conv_bn_lrelu(1024, 512, 1),
            conv_bn_lrelu(512, 1024, 3),
            conv_bn_lrelu(1024, 512, 1),
            conv_bn_lrelu(512, 1024, 3),

            conv_bn_lrelu(1024, 1024, 3),
            conv_bn_lrelu(1024, 1024, 3, s=2),

            conv_bn_lrelu(1024, 1024, 3),
            conv_bn_lrelu(1024, 1024, 3),
        )

        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(1024 * S * S, 4096)
        self.fc2 = nn.Linear(4096, S * S * (B * 5 + C))

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.dropout(F.leaky_relu(self.fc1(x), 0.1, inplace=True))
        x = self.fc2(x)
        return x.view(-1, self.S, self.S, self.B * 5 + self.C)


In [ ]:
def boxes_cxcywh_to_xyxy(b):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)

def iou_xyxy(a, b, eps=1e-9):
    inter_x1 = torch.maximum(a[...,0], b[...,0])
    inter_y1 = torch.maximum(a[...,1], b[...,1])
    inter_x2 = torch.minimum(a[...,2], b[...,2])
    inter_y2 = torch.minimum(a[...,3], b[...,3])
    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter = inter_w * inter_h
    area_a = (a[...,2]-a[...,0]).clamp(min=0) * (a[...,3]-a[...,1]).clamp(min=0)
    area_b = (b[...,2]-b[...,0]).clamp(min=0) * (b[...,3]-b[...,1]).clamp(min=0)
    union = area_a + area_b - inter
    return inter / (union + eps)

In [ ]:
class YOLOv1Loss(nn.Module):
    def __init__(self, S=7, B=2, C=20, λ_coord=5.0, λ_noobj=0.5):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        self.λ_coord, self.λ_noobj = λ_coord, λ_noobj
        self.mse = nn.MSELoss(reduction="sum")

    def forward(self, pred, target):
        Bz = pred.size(0)
        pred_boxes = pred[..., :self.B*5].view(Bz, self.S, self.S, self.B, 5)
        pred_cls = pred[..., self.B*5:]

        tgt_xy = target[..., 0:2]
        tgt_wh = target[..., 2:4]
        obj_mask = target[..., 4:5]
        tgt_cls = target[..., 5:]

        pred_xy = torch.sigmoid(pred_boxes[..., 0:2])
        pred_wh = F.relu(pred_boxes[..., 2:4])
        pred_conf = torch.sigmoid(pred_boxes[..., 4:5])

        grid_y = torch.arange(self.S, device=pred.device).view(1,self.S,1,1,1).float()
        grid_x = torch.arange(self.S, device=pred.device).view(1,1,self.S,1,1).float()
        grid_y = grid_y.expand(Bz,self.S,self.S,self.B,1)
        grid_x = grid_x.expand(Bz,self.S,self.S,self.B,1)

        abs_cx = (grid_x + pred_xy[...,0:1]) / self.S
        abs_cy = (grid_y + pred_xy[...,1:2]) / self.S
        abs_wh = pred_wh.clamp(min=1e-6)
        pred_xyxy = boxes_cxcywh_to_xyxy(torch.cat([abs_cx, abs_cy, abs_wh], dim=-1))

        tgt_cxcywh = tgt_xy.new_zeros(Bz, self.S, self.S, self.B, 4)
        tgt_cxcywh[...,0:2] = tgt_xy.unsqueeze(3)
        tgt_cxcywh[...,2:4] = tgt_wh.unsqueeze(3)
        tgt_xyxy = boxes_cxcywh_to_xyxy(tgt_cxcywh)

        ious = iou_xyxy(pred_xyxy, tgt_xyxy)
        best_iou, best_idx = ious.max(dim=3, keepdim=True)

        resp_mask = torch.zeros_like(obj_mask).expand(-1,-1,-1,self.B,-1)
        resp_mask.scatter_(3, best_idx, 1.0)
        resp_mask = resp_mask * obj_mask.unsqueeze(3)

        noobj_mask = 1.0 - resp_mask

        tgt_xy_b = tgt_xy.unsqueeze(3).expand(-1,-1,-1,self.B,-1)
        tgt_wh_b = tgt_wh.unsqueeze(3).expand(-1,-1,-1,self.B,-1)

        pred_xy_resp = pred_xy * resp_mask
        tgt_xy_resp = tgt_xy_b * resp_mask

        pred_sqrt_wh_resp = torch.sqrt(pred_wh.clamp(min=1e-6)) * resp_mask
        tgt_sqrt_wh_resp = torch.sqrt(tgt_wh_b.clamp(min=1e-6)) * resp_mask

        loss_xy = self.mse(pred_xy_resp, tgt_xy_resp)
        loss_wh = self.mse(pred_sqrt_wh_resp, tgt_sqrt_wh_resp)
        loss_coord = self.λ_coord * (loss_xy + loss_wh)

        iou_target = best_iou.unsqueeze(-1)
        loss_obj = self.mse(pred_conf * resp_mask, iou_target * resp_mask)
        loss_noobj = self.mse(pred_conf * noobj_mask, torch.zeros_like(pred_conf))
        loss_conf = loss_obj + self.λ_noobj * loss_noobj

        loss_cls = self.mse(pred_cls * obj_mask, tgt_cls * obj_mask)

        total = (loss_coord + loss_conf + loss_cls) / Bz
        return {"loss": total, "coord": loss_coord/Bz, "conf": loss_conf/Bz, "cls": loss_cls/Bz}


In [ ]:
def make_optimizer(model):
    return torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)

class YoloLRSchedule(torch.optim.lr_scheduler._LRScheduler):
    def get_lr(self):
        e = self.last_epoch
        if e <= 4:
            lr = 1e-3 + (e+1)/5 * (1e-2 - 1e-3)
        elif e <= 79:
            lr = 1e-2
        elif e <= 109:
            lr = 1e-3
        else:
            lr = 1e-4
        return [lr for _ in self.optimizer.param_groups]

In [ ]:
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

dataset = BeerYoloDataset(meta_csv="data/meta/full_dataset.csv",
                          root="data/",
                          transform=transform,
                          S=7)

loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLOv1(S=7, B=2, C=len(dataset.class_to_idx)).to(device)
criterion = YOLOv1Loss(S=7, B=2, C=len(dataset.class_to_idx),
                       λ_coord=5.0, λ_noobj=0.5)

imgs, targets = next(iter(loader))
imgs, targets = imgs.to(device), targets.to(device)
pred = model(imgs)
losses = criterion(pred, targets)
print("Loss breakdown:", {k: float(v) for k,v in losses.items()})


In [ ]:
def visualize(img_tensor, target, pred=None, S=7, C=20):
    img = img_tensor.permute(1,2,0).cpu().numpy()
    fig, ax = plt.subplots(1,1, figsize=(6,6))
    ax.imshow(img)
    H, W = img.shape[:2]

    for i in range(S):
        for j in range(S):
            if target[i,j,4] > 0.5:
                cx, cy, w, h = target[i,j,0:4]
                cx_abs = (j + cx.item()) / S * W
                cy_abs = (i + cy.item()) / S * H
                bw = w.item() * W
                bh = h.item() * H
                rect = patches.Rectangle((cx_abs - bw/2, cy_abs - bh/2),
                                         bw, bh, linewidth=2,
                                         edgecolor='lime', facecolor='none', label='GT')
                ax.add_patch(rect)

    if pred is not None:
        pred_boxes = pred[..., :B*5].view(S, S, B, 5)
        pred_cls = pred[..., B*5:]
        for i in range(S):
            for j in range(S):
                conf = pred_boxes[i,j,:,4].sigmoid()
                if conf.max() > 0.3:
                    b = conf.argmax()
                    bx, by, bw, bh = pred_boxes[i,j,b,0:4]
                    bx = bx.sigmoid().item()
                    by = by.sigmoid().item()
                    bw = F.relu(bw).item()
                    bh = F.relu(bh).item()
                    cx_abs = (j + bx) / S * W
                    cy_abs = (i + by) / S * H
                    bw_abs = bw * W
                    bh_abs = bh * H
                    rect = patches.Rectangle((cx_abs - bw_abs/2, cy_abs - bh_abs/2),
                                             bw_abs, bh_abs,
                                             linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
    plt.show()


In [ ]:
visualize(imgs[0].cpu(), targets[0].cpu(), pred[0].detach().cpu(),
          S=7, C=len(dataset.class_to_idx))

Нейронна мережа YOLO (You Only Look Once) використовує низку архітектурних параметрів та параметрів навчання (гіперпараметрів), які визначають її структуру, процес оптимізації та функцію втрат.
Ось ключові гіперпараметри та їхнє значення:
Архітектурні та структурні гіперпараметри
1. Роздільна здатність вхідного зображення (Input Resolution):
    - Значення: Під час навчання для детекції вхідна роздільна здатність мережі була збільшена з 224×224 (що використовувалася для попереднього навчання на ImageNet) до 448×448.
    - Значення: Зміна роздільної здатності дозволяє мережі використовувати дрібнозернисту візуальну інформацію, яка часто необхідна для детекції.
2. Розмір сітки (S×S):
    - Значення: Вхідне зображення поділяється на сітку розміром S×S.
    - Конкретне значення (для PASCAL VOC): S=7.
    - Значення: Якщо центр об'єкта потрапляє в певну клітинку сітки, ця клітинка стає відповідальною за детектування цього об'єкта.
3. Кількість обмежувальних рамок на клітинку (B):
    - Значення: Кожна клітинка сітки прогнозує B обмежувальних рамок (bounding boxes) та відповідні показники впевненості (confidence scores) для цих рамок.
    - Конкретне значення (для PASCAL VOC): B=2.
    - Значення: Обмежувальна рамка складається з 5 прогнозів: координати (x,y), ширина (w), висота (h) та показник впевненості (confidence).
4. Кількість класів (C):
    - Значення: Кожна клітинка сітки також прогнозує C умовних імовірностей класів, Pr(Class 
i
​
 ∣Object). Ці ймовірності обумовлені тим, що клітинка сітки містить об'єкт.
    - Конкретне значення (для PASCAL VOC): C=20.
5. (Примітка: При S=7, B=2 і C=20, кінцевий прогноз мережі має вигляд тензора 7×7×30, де 30=B×5+C=2×5+20).
Гіперпараметри функції втрат (Loss Function)
YOLO використовує суму квадратичних помилок (sum-squared error) для оптимізації, але вносить два ключові гіперпараметри для виправлення її недоліків, зокрема, нерівного вагового коефіцієнта помилок локалізації та проблем з клітинками, що не містять об'єктів.
5. Коефіцієнт координатної втрати (λ 
coord
​
 ):
    - Конкретне значення: λ 
coord
​
 =5.
    - Значення: Використовується для збільшення внеску втрат від прогнозів координат обмежувальних рамок. Це зроблено, щоб помилка локалізації не зважувалася однаково з помилкою класифікації.
6. Коефіцієнт втрати без об'єкта (λ 
noobj
​
 ):
    - Конкретне значення: λ 
noobj
​
 =0.5.
    - Значення: Використовується для зменшення внеску втрат від прогнозів впевненості для тих рамок, які не містять об'єктів. Це необхідно, оскільки у більшості клітинок сітки об'єктів немає, і якщо не зменшити цей внесок, градієнт від порожніх клітинок може переважати градієнт від клітинок з об'єктами, що може спричинити розбіжність навчання на ранніх етапах.
Гіперпараметри оптимізації та регуляризації
7. Кількість епох (Epochs):
    - Значення: Мережа навчалася приблизно 135 епох на тренувальних та валідаційних даних PASCAL VOC 2007 та 2012.
8. Розмір пакету (Batch Size):
    - Значення: Під час навчання використовувався розмір пакету 64.
9. Моментум (Momentum):
    - Значення: Використовувався моментум 0.9.
10. Згасання ваги (Weight Decay):
    - Значення: Використовувалося згасання ваги 0.0005.
11. Графік швидкості навчання (Learning Rate Schedule):
    - Значення: Швидкість навчання змінювалася протягом навчання, щоб уникнути розбіжності через нестабільні градієнти.
        ▪ Перші епохи: Повільно піднімалася з 10 
−3
  до 10 
−2
 .
        ▪ Середня фаза: Продовження навчання при 10 
−2
  протягом 75 епох.
        ▪ Пізніша фаза: Зниження до 10 
−3
  на 30 епох.
        ▪ Фінальна фаза: Зниження до 10 
−4
  на 30 епох.
12. Dropout (Виключення):
    - Конкретне значення: rate = 0.5.
    - Значення: Використовувався шар Dropout після першого повністю зв'язаного шару, щоб запобігти коадаптації між шарами та уникнути перенавчання (overfitting).
13. Аугментація даних (Data Augmentation):
    - Значення: Для уникнення перенавчання застосовувалася екстенсивна аугментація даних. Це включало випадкове масштабування та зміщення (до 20% від початкового розміру зображення). Також випадково коригувалися експозиція та насиченість зображення (до фактора 1.5) у колірному просторі HSV.